In [ ]:
!pip install qiskit
!pip install qiskit-aer
!pip install qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 8.4 MB/s eta 0:00:00


In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EnvironmentOptions, EstimatorOptions, SamplerOptions
from qiskit_aer import AerSimulator
import numpy as np
from numpy import pi
from matplotlib import pyplot as plt
import matplotlib
from scipy.optimize import minimize
from qiskit.circuit.library import QAOAAnsatz, hamiltonian_variational_ansatz, XXPlusYYGate
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.circuit import Parameter
from scipy.optimize import minimize
from scipy.linalg import eigh
from scipy.special import erf
from functools import partial
from qiskit_aer import AerSimulator
from qiskit.circuit.classical import expr

## Measurement

In [ ]:
def measure_ZZ(qc, q0, q1, cbit):
    qc.cx(q0, q1)
    qc.measure(q1, cbit)
    qc.cx(q0, q1)

In [ ]:
def measure_XI(qc, q0, q1, cbit):
    qc.h(q0)
    qc.measure(q0, cbit)
    qc.h(q0)

**Important: fix convention $ Y = S X S^\dagger$. Later formulas must use the same convention to ensure a correct pauli tracking**

In [ ]:
def measure_YI(qc, q0, q1, cbit):
    qc.sdg(q0)
    measure_XI(qc, q0, q1, cbit)
    qc.s(q0)

In [ ]:
def measure_ZY(qc, q0, q1, cbit):

    # with the same convention, Y = S X S^d = S H Z H S^d
    qc.sdg(q1)
    qc.h(q1)
    measure_ZZ(qc, q0, q1, cbit)
    qc.h(q1)
    qc.s(q1)

## Single qubit clifford group

In [ ]:
def add_S(qc, q):
    c = ClassicalRegister(4)
    qc.add_register(c)

    # (ancilla, data) = (q+1, q) initilize ancilla qubit in |+>
    qc.h(q+1)

    measure_XI(qc, q+1, q, c[0])
    measure_ZZ(qc, q+1, q, c[1])
    measure_YI(qc, q+1, q, c[2])
    measure_XI(qc, q+1, q, c[3])

    # measurement bit c[i] encodes s_i = (-1)^{c[i]}
    #   s_i s_j = (-1)^{c[i] + c[j]}
    #   → product becomes XOR at bit level

    # --- Z^{(1 + s0 s1 s2)/2} ---
    # exponent = 1 when s0 s1 s2 = +1
    # s0 s1 s2 = (-1)^{c0 + c1 + c2}
    # → +1 when (c0 + c1 + c2) mod 2 = 0 (even parity)

    parity = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[2])
    with qc.if_test(expr.logic_not(parity)):
        qc.z(q)

    qc.reset(q+1)

    return qc

In [ ]:
def add_HSH(qc, q):
    c = ClassicalRegister(4)
    qc.add_register(c)

    qc.h(q+1)

    measure_XI(qc, q+1, q, c[0])
    measure_ZZ(qc, q+1, q, c[1])
    measure_ZY(qc, q+1, q, c[2])
    measure_XI(qc, q+1, q, c[3])

    # X^{(1 + s1 s2)/2} (even parity)

    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(q)

    # Y^{(1 - s0 s3)/2} (odd parity)

    parity_03 = expr.bit_xor(c[0], c[3])
    with qc.if_test(parity_03):
        qc.y(q)

    qc.reset(q+1)

    return qc

In [ ]:
def add_HS(qc, q):
    c = ClassicalRegister(5)
    qc.add_register(c)

    qc.h(q+1)

    measure_XI(qc, q+1, q, c[0])  # s0
    measure_ZY(qc, q+1, q, c[1])  # s1
    measure_ZZ(qc, q+1, q, c[2])  # s2
    measure_YI(qc, q+1, q, c[3])  # s3
    measure_XI(qc, q+1, q, c[4])  # s4

    # Z^{(1 + s0 s1 s3)/2} → even parity of 3 bits
    parity_013 = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[3])
    with qc.if_test(expr.logic_not(parity_013)):
        qc.z(q)

    # X^{(1 + s1 s2)/2} → even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(q)

    qc.reset(q+1)

    return qc

## Single qubit clifford gate in supremacy circuit

In [ ]:
def add_sqrtX(qc, q):
    add_HSH(qc, q)
    return qc

In [ ]:
def add_sqrtY(qc, q):
    add_S(qc, q)
    add_HS(qc, q)
    return qc

## Test on $|0\rangle$ and $|+\rangle$ to cover the whole Hilbert space of qubit

In [ ]:
# sqrt(X) test — input |0>
# remember sqrt(X) rotates Z to -Y, final state is |-y>

# 1) direct sqrt(X)
qc1 = QuantumCircuit(2)
qc1.sx(0)

sv1 = Statevector.from_instruction(qc1)

print("direct sqrt(X) on |0>:")
print(sv1.data)


# 2) MBQC sqrt(X)
qc2 = QuantumCircuit(2)
add_sqrtX(qc2, 0)

qc2.save_statevector(conditional=True)

sim = AerSimulator(method="statevector")
result = sim.run(qc2, shots=1).result()

print("\ngadget result:")
print(result.data(0)["statevector"])

direct sqrt(X) on |0>:
[0.5+0.5j 0.5-0.5j 0. +0.j  0. +0.j ]

gadget result:
{'0x6': Statevector([1.29893408e-16+7.07106781e-01j,
             7.07106781e-01-1.29893408e-16j,
             0.00000000e+00+0.00000000e+00j,
             0.00000000e+00+0.00000000e+00j],
            dims=(2, 2))}


In [ ]:
# sqrt(X) test — input |+>

# 1) direct sqrt(X)
qc1 = QuantumCircuit(2)
qc1.h(0)
qc1.sx(0)

sv1 = Statevector.from_instruction(qc1)

print("direct sqrt(X) on |+>:")
print(sv1.data)


# 2) MBQC sqrt(X)
qc2 = QuantumCircuit(2)
qc2.h(0)
add_sqrtX(qc2, 0)

qc2.save_statevector(conditional=True)

result = sim.run(qc2, shots=1).result()

print("\ngadget result:")
print(result.data(0)["statevector"])

direct sqrt(X) on |+>:
[0.70710678+0.j 0.70710678+0.j 0.        +0.j 0.        +0.j]

gadget result:
{'0xa': Statevector([-0.5+0.5j, -0.5+0.5j,  0. +0.j ,  0. +0.j ],
            dims=(2, 2))}


In [ ]:
# sqrt(Y) test — input |0>
# remember sqrt(Y) rotates Z to X, final state is |+>

# 1) direct sqrt(Y)
qc1 = QuantumCircuit(2)
qc1.ry(np.pi/2, 0)

sv1 = Statevector.from_instruction(qc1)

print("direct sqrt(Y) on |0>:")
print(sv1.data)


# 2) MBQC sqrt(Y)
qc2 = QuantumCircuit(2)
add_sqrtY(qc2, 0)

qc2.save_statevector(conditional=True)

sim = AerSimulator(method="statevector")
result = sim.run(qc2, shots=1).result()

print("\ngadget result:")
print(result.data(0)["statevector"])

direct sqrt(Y) on |0>:
[0.70710678+0.j 0.70710678+0.j 0.        +0.j 0.        +0.j]

gadget result:
{'0x10a': Statevector([ 0.70710678-5.96875655e-16j,  0.70710678-4.79118720e-16j,
             -0.        +0.00000000e+00j, -0.        +0.00000000e+00j],
            dims=(2, 2))}


In [ ]:
# sqrt(Y) test — input |+>

# 1) direct sqrt(Y)
qc1 = QuantumCircuit(2)
qc1.h(0)
qc1.ry(np.pi/2, 0)

sv1 = Statevector.from_instruction(qc1)

print("direct sqrt(Y) on |+>:")
print(sv1.data)


# 2) MBQC sqrt(Y)
qc2 = QuantumCircuit(2)
qc2.h(0)
add_sqrtY(qc2, 0)

qc2.save_statevector(conditional=True)

result = sim.run(qc2, shots=1).result()

print("\ngadget result:")
print(result.data(0)["statevector"])

direct sqrt(Y) on |+>:
[8.86511593e-17+0.j 1.00000000e+00+0.j 0.00000000e+00+0.j
 0.00000000e+00+0.j]

gadget result:
{'0x106': Statevector([ 5.55111512e-17-5.55111512e-17j,
             -1.00000000e+00+7.50501663e-16j,
             -0.00000000e+00+0.00000000e+00j,
              0.00000000e+00+0.00000000e+00j],
            dims=(2, 2))}


## Some other intermediate test. No need to read

In [ ]:
# S test

# 1) direct S gate
qc1 = QuantumCircuit(2)
qc1.h(0)     # data in |+>
qc1.s(0)     # direct S on data
sv1 = Statevector.from_instruction(qc1)

print("direct S final vector:")
print(sv1.data)


# 2) MBQC S gate
qc2 = QuantumCircuit(2)
qc2.h(0)     # same input on data
add_S(qc2, 0)
qc2.save_statevector(conditional=True)

sim = AerSimulator(method="statevector")
result = sim.run(qc2, shots=1).result()

print("\ngadget final vector(s):")
print(result.data(0)["statevector"])

direct S final vector:
[0.70710678+0.j         0.        +0.70710678j 0.        +0.j
 0.        +0.j        ]

gadget final vector(s):
{'0x8': Statevector([-0.5+0.5j, -0.5-0.5j,  0. +0.j ,  0. +0.j ],
            dims=(2, 2))}


In [ ]:
# HSH test

qc1 = QuantumCircuit(2)
qc1.h(0)         # data in |+>
qc1.h(0)
qc1.s(0)
qc1.h(0)

sv1 = Statevector.from_instruction(qc1)

print("direct HSH final vector:")
print(sv1.data)

qc2 = QuantumCircuit(2)
qc2.h(0)         # same input
add_HSH(qc2, 0)

qc2.save_statevector(conditional=True)

sim = AerSimulator(method="statevector")
result = sim.run(qc2, shots=1).result()

print("\ngadget final vector(s):")
print(result.data(0)["statevector"])

direct HSH final vector:
[0.70710678-1.5818787e-17j 0.70710678+1.5818787e-17j
 0.        +0.0000000e+00j 0.        +0.0000000e+00j]

gadget final vector(s):
{'0x4': Statevector([0.5-0.5j, 0.5-0.5j, 0. +0.j , 0. +0.j ],
            dims=(2, 2))}


In [ ]:
# HS test

qc1 = QuantumCircuit(2)

qc1.h(0)      # prepare |+>
qc1.s(0)
qc1.h(0)      # apply HS

sv1 = Statevector.from_instruction(qc1)

print("direct HS final vector:")
print(sv1.data)


qc2 = QuantumCircuit(2)

qc2.h(0)      # same input
add_HS(qc2, 0)

qc2.save_statevector(conditional=True)

sim = AerSimulator(method="statevector")
result = sim.run(qc2, shots=1).result()

print("\ngadget final vector(s):")
print(result.data(0)["statevector"])

direct HS final vector:
[0.5+0.5j 0.5-0.5j 0. +0.j  0. +0.j ]

gadget final vector(s):
{'0xe': Statevector([-2.74766180e-16-7.07106781e-01j,
             -7.07106781e-01+2.74766180e-16j,
              0.00000000e+00-0.00000000e+00j,
             -0.00000000e+00+0.00000000e+00j],
            dims=(2, 2))}


In [ ]:
# parity measurement test

def measure_ZZ(qc, q0, q1, cbit):
    qc.cx(q0, q1)
    qc.measure(q1, cbit)
    qc.cx(q0, q1)

# two data qubits, one classical bit for the ZZ result
q = QuantumRegister(2, "q")
c = ClassicalRegister(1, "c")
qc = QuantumCircuit(q, c)

# prepare |++> = (|00> + |01> + |10> + |11>) / 2
qc.h(q[0])
qc.h(q[1])

# do the ZZ measurement gadget
measure_ZZ(qc, q[0], q[1], c[0])

# save the post-measurement state, split by classical outcome
qc.save_density_matrix(label="rho", conditional=True)

print(qc.draw())

sim = AerSimulator(method="density_matrix")
result = sim.run(qc, shots=2000).result()

counts = result.get_counts()
saved = result.data(0)["rho"]

print("counts =", counts)
print("\nSaved conditional density matrices:\n")
for key, rho in saved.items():
    print(f"classical outcome {key}:")
    print(np.array(rho))
    print()

     ┌───┐              rho 
q_0: ┤ H ├──■───────■────░──
     ├───┤┌─┴─┐┌─┐┌─┴─┐  ░  
q_1: ┤ H ├┤ X ├┤M├┤ X ├──░──
     └───┘└───┘└╥┘└───┘  ░  
c: 1/═══════════╩═══════════
                0           
counts = {'1': 1009, '0': 991}

Saved conditional density matrices:

classical outcome 0x1:
[[0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0.5+0.j 0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j 0.5+0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]]

classical outcome 0x0:
[[0.5+0.j 0. +0.j 0. +0.j 0.5+0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0.5+0.j 0. +0.j 0. +0.j 0.5+0.j]]



In [ ]:
# bit_xor test

q = QuantumRegister(3)
c = ClassicalRegister(3)

qc = QuantumCircuit(q, c)

# Put q0, q1 in superposition → all 4 outcomes equally likely
qc.h(0)
qc.h(1)

# Measure them
qc.measure(q[0], c[0])   # b0
qc.measure(q[1], c[1])   # b1

# XOR condition
parity = expr.bit_xor(c[0], c[1])

# Flip q2 if XOR = 1
with qc.if_test(parity):
    qc.x(2)

# Measure target qubit
qc.measure(q[2], c[2])

# Run
sim = AerSimulator()
result = sim.run(qc, shots=1000).result()
counts = result.get_counts()

print(counts)